# 🧠 Limpar sinais do cérebro: remover artefactos do EEG

Bem-vindos! Neste notebook vamos pegar num registo real da atividade do cérebro e **limpá-lo**, tal como os investigadores fazem antes de analisarem os seus dados.

**Não é preciso saber programar.** Cada caixa cinzenta abaixo é um pequeno pedaço de código. Para o correr:
- clique na caixa e carregue em **Shift + Enter** (ou no botão ▶).
- espere até que o `[ ]` à esquerda mostre um número: isso quer dizer que terminou.
- corra as caixas **por ordem**, de cima para baixo.

### O que vamos fazer hoje
1. 👀 **Conhecer os dados**: como é um registo de EEG?
2. 🔍 **Encontrar os "artefactos"**: sinais que *não* vêm do cérebro
3. 🎚️ **Filtrar**: remover oscilações lentas e ruído rápido
4. 🔌 **Um elétrodo avariado**: como o reparar
5. 👁️ **O pestanejar**: o maior artefacto do EEG
6. 🍸 **ICA, o truque da "festa barulhenta"**: separar o pestanejar da atividade do cérebro
7. ✨ **Antes vs. depois**: resultou?

Vamos usar o **[MNE-Python](https://mne.tools)**, uma ferramenta gratuita e muito utilizada para analisar sinais do cérebro.

## 0. Preparação

As duas caixas seguintes instalam e carregam as ferramentas de que precisamos. Pode demorar cerca de um minuto.

In [ ]:
# Instalar o MNE no Colab. Escondemos as mensagens do pip, porque o Colab mostra avisos
# inofensivos ("dependency conflicts") sobre pacotes que não usamos.
import subprocess, sys
print("A instalar o MNE...")
result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "mne"],
                        capture_output=True, text=True)
print("Pronto ✅" if result.returncode == 0 else result.stderr)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import mne
from mne.datasets import eegbci

mne.set_log_level("ERROR")                 # manter as mensagens arrumadas
mne.viz.set_browser_backend("matplotlib")  # imagens estáticas
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("data/raw")                 # onde ficam os registos descarregados
RAW_DIR.mkdir(parents=True, exist_ok=True)

### Definições

O registo que vamos usar vem de um conjunto de dados público ([PhysioNet EEG Motor Movement/Imagery](https://physionet.org/content/eegmmidb/1.0.0/)): 109 voluntários usaram uma touca de EEG enquanto abriam e fechavam as mãos. Cada voluntário tem um número.

👉 **Mais tarde, pode mudar o `SUBJECT` para ver outra pessoa** (qualquer número de 1 a 109).

In [ ]:
SUBJECT = 1   # qual o voluntário (1 a 109)
RUN = 4       # qual a sessão de registo (4 = uma tarefa de 2 minutos)

## 1. 👀 Conhecer os dados

O **EEG** (eletroencefalografia) mede sinais elétricos minúsculos do cérebro, através de elétrodos colocados no couro cabeludo. Estes sinais são *mesmo* pequenos: alguns milionésimos de volt (**microvolts, µV**). Para comparar, uma pilha AA dá 1,5 volts, cerca de um milhão de vezes mais.

Vamos carregar um registo.

In [ ]:
fname = eegbci.load_data(SUBJECT, [RUN], path=RAW_DIR, update_path=False)[0]
raw = mne.io.read_raw_edf(fname, preload=True)
eegbci.standardize(raw)             # arrumar os nomes dos elétrodos
raw.set_montage("standard_1005")    # dizer ao MNE onde está cada elétrodo na cabeça
raw.set_annotations(None)           # retirar as marcações da tarefa: hoje não precisamos delas

print(f"Número de elétrodos   : {len(raw.ch_names)}")
print(f"Duração do registo    : {raw.times[-1]:.0f} segundos")
print(f"Medições por segundo  : {raw.info['sfreq']:.0f} (em cada elétrodo)")

São 64 elétrodos, cada um a medir o cérebro **160 vezes por segundo**.

Onde estão estes elétrodos? Cada ponto abaixo é um elétrodo, visto de cima (o nariz aponta para cima). Os nomes seguem uma convenção: **F** = frontal (testa), **C** = central, **P** = parietal, **O** = occipital (parte de trás da cabeça), **T** = temporal (perto das orelhas). Os elétrodos **Fp** ficam mesmo por cima dos olhos. Não se esqueçam disto!

In [ ]:
raw.plot_sensors(show_names=True, sphere="eeglab");

## 2. 🔍 Olhar para o sinal em bruto

Agora vamos ver o próprio sinal. Cada linha é um elétrodo, e o tempo avança da esquerda para a direita. Mostramos alguns elétrodos, da testa (**em cima**) até à parte de trás da cabeça (**em baixo**), durante 10 segundos.

In [ ]:
# Alguns elétrodos, da frente para trás
SHOW = ["Fp1", "Fp2", "AF3", "F3", "Fz", "C3", "Cz", "P3", "Pz", "O1", "Oz"]

# Encontrar os pestanejos e escolher os 10 segundos com mais pestanejos, para haver algo para ver
blinks = mne.preprocessing.find_eog_events(raw.copy().filter(1, 15), ch_name=["Fp1", "Fp2"])
BLINK_TIMES = blinks[:, 0] / raw.info["sfreq"] - raw.first_time
START = max(np.arange(0, raw.times[-1] - 10, 0.5),
            key=lambda s: np.sum((BLINK_TIMES > s + 1) & (BLINK_TIMES < s + 9)))


def show(recording, title, start=START, duration=10, channels=SHOW, spacing=150, mark_blinks=False):
    # Desenhar sempre os mesmos elétrodos e o mesmo intervalo de tempo, para ser fácil comparar
    idx = [recording.ch_names.index(ch) for ch in channels]
    data, times = recording.get_data(picks=idx, tmin=start, tmax=start + duration, return_times=True)
    data = (data - data.mean(axis=1, keepdims=True)) * 1e6   # volts -> microvolts, centrados
    fig, ax = plt.subplots(figsize=(12, 0.45 * len(channels) + 1.2))
    for i, trace in enumerate(data):
        ax.plot(times, trace - i * spacing, color="black", lw=0.7)
    if mark_blinks:
        for t in BLINK_TIMES[(BLINK_TIMES > start) & (BLINK_TIMES < start + duration)]:
            ax.axvspan(t - 0.2, t + 0.2, color="tab:blue", alpha=0.15)
    ax.set_yticks(-np.arange(len(channels)) * spacing, channels)
    ax.set_ylim(-len(channels) * spacing, 1.5 * spacing)
    ax.set_xlim(times[0], times[-1])
    ax.set_xlabel("Tempo (segundos)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.spines[["top", "right", "left"]].set_visible(False)
    # Uma barra de escala, para se ver o tamanho de 100 microvolts
    ax.plot([times[-1]] * 2, [spacing * 0.3, spacing * 0.3 + 100], color="tab:red", lw=3, clip_on=False)
    ax.text(times[-1], spacing * 0.3 + 50, "  100 µV", color="tab:red", va="center")
    plt.show()


def compare(before, after, title, channel="Fp1", start=START, duration=10):
    # Desenhar um elétrodo antes (vermelho) e depois (preto) da limpeza, sobrepostos
    fig, ax = plt.subplots(figsize=(12, 3))
    for recording, color, label in [(before, "tab:red", "antes"), (after, "black", "depois")]:
        data, times = recording.get_data(picks=[channel], tmin=start, tmax=start + duration,
                                         return_times=True)
        ax.plot(times, data[0] * 1e6, color=color, lw=0.9, label=label)
    ax.set_xlabel("Tempo (segundos)")
    ax.set_ylabel(f"{channel} (µV)")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.legend(loc="upper right")
    ax.spines[["top", "right"]].set_visible(False)
    plt.show()


def spectrum(recording, title):
    # Quanto existe de cada frequência no registo (média de todos os elétrodos)
    fig = recording.compute_psd(fmax=80).plot(average=True, amplitude=False, show=False)
    ax = fig.axes[0]
    ax.set_title("")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.set_xlabel("Frequência (Hz)")
    ax.set_ylabel("Potência (dB)")
    plt.show()


show(raw, "Sinal em bruto, tal como sai do amplificador")

Olhem para as **linhas de cima (Fp1, Fp2)**: veem os grandes "montes" que se destacam do resto? Vamos realçá-los:

In [ ]:
show(raw, "O mesmo sinal, com os pestanejos realçados", mark_blinks=True)

São **pestanejos!** 👁️ São muito maiores do que a atividade do cérebro, mas não vêm do cérebro: o olho funciona como uma pequena pilha e, sempre que a pálpebra se mexe, altera o campo elétrico perto da testa.

A sinais deste tipo chamamos **artefactos**: coisas que os elétrodos captam mas que **não vêm do cérebro**. Os investigadores têm de os remover, senão podiam confundir um pestanejo com atividade do cérebro.

| Artefacto | De onde vem | Como se parece |
|---|---|---|
| 👁️ Pestanejar | o olho e a pálpebra | "montes" grandes e lentos na testa |
| 👀 Movimentos dos olhos | olhar para a esquerda/direita | mudanças em degrau perto dos olhos |
| 😬 Músculos (maxilar, pescoço) | cerrar os dentes, mastigar, tensão | atividade rápida e "peluda" |
| ❤️ Batimento cardíaco | a atividade elétrica do coração | pequenos picos regulares |
| 🔌 Rede elétrica | as tomadas (50 ou 60 Hz) | um zumbido rápido e constante |
| 💧 Suor, movimento | a pele e os cabos | oscilações lentas, à deriva |
| ⚡ Elétrodo avariado | mau contacto com o couro cabeludo | uma linha plana ou com imenso ruído |

💡 *Ideia para uma demonstração ao vivo: se tiverem uma touca de EEG, peçam a um voluntário para pestanejar, cerrar os dentes ou mexer os olhos, e vejam estes padrões a aparecer.*

## 3. 🎚️ Filtrar: remover oscilações lentas e ruído rápido

Os sinais do cérebro são feitos de **ritmos**, ondas lentas e rápidas misturadas, como os instrumentos numa música. A velocidade de uma onda chama-se **frequência** e mede-se em **Hertz (Hz)**: 10 Hz quer dizer 10 ondas por segundo.

O gráfico abaixo mostra **quanto existe de cada frequência** no nosso registo (um pouco como o equalizador de uma aparelhagem de música).

In [ ]:
spectrum(raw, "Frequências no registo em bruto")

A maior parte da atividade interessante do cérebro está, mais ou menos, **entre 1 e 40 Hz**:
- **abaixo de 1 Hz** há sobretudo oscilações lentas (suor, pequenos movimentos dos cabos)
- **acima de 40 Hz** há sobretudo atividade dos músculos e ruído elétrico
- 🔌 veem o **pico estreito nos 60 Hz**? É a **rede elétrica**! A eletricidade das tomadas oscila 60 vezes por segundo (estes dados foram registados nos EUA; em Portugal e no resto da Europa são 50 Hz), e os elétrodos captam-na como se fossem uma antena.

Um **filtro** funciona como o equalizador da aparelhagem: baixamos as frequências que não queremos e mantemos as que queremos. Aqui mantemos **de 1 a 40 Hz**.

In [ ]:
filtered = raw.copy().filter(l_freq=1.0, h_freq=40.0)

spectrum(filtered, "Frequências depois de filtrar (1-40 Hz)")

Tudo o que está acima de 40 Hz, incluindo o pico da rede elétrica nos 60 Hz, foi atenuado. E como fica o sinal agora?

In [ ]:
show(raw, "ANTES de filtrar")
show(filtered, "DEPOIS de filtrar (1-40 Hz)")

As linhas estão **mais suaves e mais estáveis**. Mas reparem: **os pestanejos continuam lá!** 😕

Isto acontece porque os pestanejos são ondas lentas, na *mesma* gama de frequências da atividade do cérebro. Um filtro não os consegue separar. Vamos precisar de um truque mais inteligente (secção 5). Mas antes, um pequeno desvio.

## 4. 🔌 Um elétrodo avariado

Às vezes um elétrodo perde o contacto com o couro cabeludo (gel seco, cabelo pelo meio, um cabo solto). Nesse caso regista sobretudo ruído.

O nosso registo está bastante limpo, por isso **vamos avariar um de propósito** 😈: acrescentamos muito ruído ao elétrodo **Cz** (no topo da cabeça).

In [ ]:
broken = filtered.copy()
rng = np.random.default_rng(0)
cz = broken.ch_names.index("Cz")
broken._data[cz] += rng.normal(0, 60e-6, broken.n_times)   # acrescentar muito ruído ao Cz

show(broken, "O Cz tem mau contacto")

O **Cz** está claramente avariado. O que podemos fazer?

1. Dizer ao MNE que este elétrodo é **mau**.
2. **Interpolá-lo**: estimar o que ele devia ter registado a partir dos **vizinhos**. É como adivinhar a temperatura de uma vila a partir das temperaturas das vilas à volta.

In [ ]:
broken.info["bads"] = ["Cz"]                   # 1. marcá-lo como mau
repaired = broken.copy().interpolate_bads()    # 2. reconstruí-lo a partir dos vizinhos

show(repaired, "Cz reconstruído a partir dos vizinhos")

O Cz voltou a parecer um elétrodo normal. 🎉

(Continuamos com o registo `filtered`, que nunca teve este problema.)

## 5. 👁️ O pestanejar: o maior artefacto

Vamos estudar os pestanejos com mais atenção. O MNE consegue **encontrar todos os pestanejos** automaticamente, procurando os "montes" grandes nos elétrodos da testa, e depois **fazer a média de todos** para mostrar como é um "pestanejo típico".

(Estes dados não têm um elétrodo dedicado aos olhos, por isso usamos o **Fp1** e o **Fp2**, os elétrodos mesmo por cima dos olhos.)

In [ ]:
blink_epochs = mne.preprocessing.create_eog_epochs(filtered, ch_name=["Fp1", "Fp2"],
                                                    baseline=(-0.5, -0.2))
print(f"Número de pestanejos encontrados: {len(blink_epochs)}")

average_blink = blink_epochs.average()
fig = average_blink.plot_joint(title="O pestanejo médio", times=[0.0], ts_args=dict(gfp=False), show=False)
for ax in fig.axes:
    if ax.get_xlabel() == "Time (s)":
        ax.set_xlabel("Tempo (s)")
plt.show()

Como ler esta imagem:
- **as linhas** são os elétrodos. No instante **0** acontece o pestanejo, e o sinal fica muito grande.
- **o mapa da cabeça** mostra *onde* na cabeça o pestanejo é mais forte. **Vermelho = forte**. Está tudo na **frente**, perto dos olhos, e vai desaparecendo para trás.

**Esta "forma na cabeça" é a chave para remover os pestanejos**, como vamos ver a seguir.

## 6. 🍸 ICA: o truque da "festa barulhenta"

Imaginem que estão numa festa 🎉 com **várias pessoas a falar ao mesmo tempo**, e que espalharam **vários microfones** pela sala. Cada microfone grava uma **mistura** de todas as vozes, mais alta para as pessoas que estão mais perto dele.

Existe um método matemático chamado **Análise de Componentes Independentes (ICA**, do inglês *Independent Component Analysis*) que consegue pegar em todas essas gravações e **voltar a separar cada uma das vozes**.

O EEG é exatamente a mesma situação:
- 🗣️ as **"vozes"** são as diferentes fontes de atividade: diferentes zonas do cérebro... e os **olhos**, os **músculos**, o **coração**!
- 🎤 os **"microfones"** são os nossos 64 elétrodos, e cada um grava uma mistura de tudo

Então o plano é:
1. **Separar** o sinal nas suas "vozes" (chamadas **componentes**)
2. **Encontrar a voz que pertence aos pestanejos**
3. **Silenciá-la**, e juntar tudo o resto outra vez

```
 elétrodos (misturados)       componentes (separados)          elétrodos (limpos)
  Fp1  ~~~^~~~^~~               cérebro   ~~~~~~~~~                 Fp1  ~~~~~~~~~
  Fz   ~~~^~~~^~~  ── ICA ──▶   cérebro   ~~~~~~~~~  ── silenciar ─▶ Fz   ~~~~~~~~~
  Oz   ~~~~~~~~~~               pestanejo ___^___^_   👁️              Oz   ~~~~~~~~~
```

Vamos separar os componentes. Demora apenas alguns segundos.

In [ ]:
ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter="auto")
ica.fit(filtered)
print("Feito! O sinal foi separado em", ica.n_components_, "componentes.")

Separámos o sinal em **20 componentes**. Esta é a **"forma na cabeça"** de cada um.

🕵️ **A vossa vez: qual destes componentes se parece com o pestanejo?** Lembrem-se do pestanejo médio da secção 5: forte na frente, perto dos olhos.

In [ ]:
ica.plot_components(sphere="eeglab");

Encontraram? Vamos confirmar com o MNE: ele compara cada componente com os elétrodos da testa e dá uma **pontuação** (perto de 1 ou -1 = muito parecido com os pestanejos, perto de 0 = sem relação). Há um gráfico para cada elétrodo da testa (Fp1 em cima, Fp2 em baixo), e o componente escolhido aparece a **vermelho**.

In [ ]:
blink_components, scores = ica.find_bads_eog(filtered, ch_name=["Fp1", "Fp2"])
print("Componente(s) que se parecem com pestanejos:", [int(c) for c in blink_components])

fig = ica.plot_scores(scores, exclude=blink_components, show=False,
                      title="Quão parecido é cada componente com os pestanejos?")
for ax in fig.axes:
    ax.set_xlabel("Componente")
    ax.set_ylabel("Pontuação")
plt.show()

Vamos também ver a **evolução no tempo** de alguns componentes. O componente do pestanejo deve mostrar um "monte" sempre que a pessoa pestaneja, enquanto os outros se parecem com atividade contínua do cérebro.

In [ ]:
fig = ica.plot_sources(filtered, picks=range(5), start=START, stop=START + 10,
                       show_scrollbars=False, show=False)
fig.mne.ax_main.set_xlabel("Tempo (s)")
plt.show()

O **ICA000** tem uma descida acentuada em cada pestanejo, enquanto os outros parecem atividade contínua. (Aqui os pestanejos apontam *para baixo* em vez de para cima: é normal, para a ICA tanto faz cima ou baixo.)

## 7. ✨ Remover os pestanejos e comparar

Agora **silenciamos** o componente do pestanejo e reconstruímos os sinais dos elétrodos com todos os outros.

In [ ]:
ica.exclude = blink_components
cleaned = ica.apply(filtered.copy())

show(filtered, "ANTES: filtrado, os pestanejos continuam lá", mark_blinks=True)
show(cleaned, "DEPOIS: componente do pestanejo removido", mark_blinks=True)

Os pestanejos desapareceram do Fp1 e do Fp2, e **o resto do sinal continua lá**. 🎉

Aqui está o elétrodo da testa **Fp1** numa só imagem (**vermelho = antes, preto = depois**):

In [ ]:
compare(filtered, cleaned, "Fp1, antes e depois de remover os pestanejos")

Uma última verificação: o **pestanejo médio**, antes e depois da limpeza (com a mesma escala nos dois).

In [ ]:
cleaned_blink = mne.Epochs(cleaned, blink_epochs.events, tmin=-0.5, tmax=0.5,
                           baseline=(-0.5, -0.2)).average()

ylim = dict(eeg=[-150, 250])
for evoked, title in [(average_blink, "Pestanejo médio ANTES da limpeza"),
                      (cleaned_blink, "Os mesmos momentos DEPOIS da limpeza")]:
    fig = evoked.plot(ylim=ylim, titles=title, gfp=False, show=False)
    fig.axes[0].set_xlabel("Tempo (s)")
    fig.axes[0].set_title(title)
    plt.show()

Nos momentos em que a pessoa pestanejou, agora **quase não sobra nada**. ✅

## 🎓 Resumo

| Problema | Ferramenta | Ideia |
|---|---|---|
| Oscilações lentas, ruído rápido | **Filtro** | manter só as frequências do "cérebro" (1-40 Hz) |
| Um elétrodo avariado | **Interpolação** | reconstruí-lo a partir dos vizinhos |
| Pestanejos | **ICA** | separar as "vozes" e silenciar a do pestanejo |

A limpeza é normalmente o **primeiro passo** de qualquer estudo de EEG. Só depois disto é que os investigadores começam a fazer perguntas sobre o próprio cérebro.

## 🧪 Experimentem vocês

1. Voltem às **Definições**, mudem o `SUBJECT` para outro número (por exemplo `3` ou `7`) e corram tudo outra vez (**Runtime → Run all** no Colab, **Run → Run All Cells** no Jupyter). Os pestanejos são iguais em todas as pessoas?
2. Na secção 7, removam um componente **errado**, por exemplo `ica.exclude = [5]`. O que acontece ao sinal?
3. Na secção 4, "avariem" outro elétrodo, como o `"Oz"` ou o `"Fp1"`.

---
### 📁 Para os organizadores: usar os vossos próprios registos

Coloquem os ficheiros na pasta `data/raw/` e substituam a caixa de carregamento da secção 1 por:

```python
raw = mne.io.read_raw(RAW_DIR / "o_meu_registo.edf", preload=True)   # .edf, .bdf, .fif, .set, .vhdr ...
raw.set_montage("standard_1005")
```

Se o vosso sistema tiver um elétrodo dedicado aos olhos (muitas vezes chamado `EOG` ou `VEOG`), usem-no em vez de `["Fp1", "Fp2"]` nas secções 5 e 6. Para calcular a ICA com antecedência, vejam o `scripts/prepare_data.py`.